## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-12-18-54-00 +0000,bbc,Two men guilty of spying for Iran over plan to...,https://www.bbc.co.uk/news/articles/ck9q8g8209...
1,2026-11-12-18-53-31 +0000,nypost,Partner of Alaska dad fatally struck by Hollyw...,https://nypost.com/2026/11/12/us-news/partner-...
2,2026-11-12-18-52-59 +0000,startribune,"Dave Cossetta, whose family name defined St. P...",https://www.startribune.com/legendary-st-paul-...
3,2026-11-12-18-52-00 +0000,wsj,Another Winner From the Midterms: Crusaders Ag...,https://www.wsj.com/finance/another-winner-fro...
4,2026-11-12-18-51-48 +0000,nyt,Trump Says He’s Pulling Vance Off Iran Talks t...,https://www.nytimes.com/2026/11/12/us/politics...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2673/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
118,huang,32
58,nvidia,32
117,jensen,31
43,trump,27
162,new,27


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
337,2026-11-11-22-25-23 +0000,nypost,Nvidia’s Jensen Huang also had alleged affair ...,https://nypost.com/2026/11/11/business/nvidias...,186
134,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...,181
105,2026-11-12-15-50-09 +0000,nypost,Trump floats ex-Apple CEO Tim Cook to succeed ...,https://nypost.com/2026/11/12/us-news/trump-fl...,158
157,2026-11-12-13-56-33 +0000,nypost,Jensen Huang quits as Nvidia CEO in wake of Ch...,https://nypost.com/2026/11/12/business/jensen-...,147
154,2026-11-12-13-59-07 +0000,chitri,Nvidia CEO Jensen Huang resigns after his affa...,https://www.chicagotribune.com/2026/11/12/nvid...,144


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
337,186,2026-11-11-22-25-23 +0000,nypost,Nvidia’s Jensen Huang also had alleged affair ...,https://nypost.com/2026/11/11/business/nvidias...
113,140,2026-11-12-15-15-00 +0000,wsj,President Trump said the U.S. doesn’t need a d...,https://www.wsj.com/politics/policy/trump-now-...
264,105,2026-11-12-04-02-00 +0000,wsj,Democrat Scott Colom is the state of Mississip...,https://www.wsj.com/politics/elections/scott-c...
26,94,2026-11-12-18-17-00 +0000,wsj,Pentagon chief Pete Hegseth pinned a medal on ...,https://www.wsj.com/politics/hegseth-and-aides...
280,60,2026-11-12-02-00-00 +0000,wsj,“The ass-kicking we needed.” South Carolina Re...,https://www.wsj.com/politics/elections/republi...
203,60,2026-11-12-11-00-00 +0000,latimes,"L.A. sales taxes will go up, but the time it t...",https://www.latimes.com/california/story/2026-...
298,57,2026-11-12-00-24-23 +0000,chitri,Chicago parents and students protest Google’s ...,https://www.chicagotribune.com/2026/11/11/cps-...
328,56,2026-11-11-22-39-00 +0000,wsj,U.S. Treasury Secretary Scott Bessent was rece...,https://www.wsj.com/politics/national-security...
93,56,2026-11-12-16-29-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...
77,46,2026-11-12-17-13-14 +0000,bbc,Washington DC shooting leaves one dead and fou...,https://www.bbc.co.uk/news/articles/crgqx2qqnw...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
